# Lab 3.6 - Amazon SageMaker: Generating model performance metrics

**Educate edition.** Replaces `en_us/3_6-machinelearning.ipynb`.

## Objectives
* Use the test data to generate predictions
* Generate a confusion matrix from the results
* Generate performance metrics for the model

**Prerequisite:** run Labs 3.4 and 3.5.

**Cost note:** this lab is pure analysis. It uses no AWS services
beyond the notebook instance in either track.

## Lab configuration - CHOOSE YOUR TRACK

This notebook runs in one of two modes. Set the flag in the next cell.

| | `USE_MANAGED_SAGEMAKER = False` (**Track B**) | `USE_MANAGED_SAGEMAKER = True` (**Track A**) |
|---|---|---|
| Where training runs | Inside this notebook | A separate managed SageMaker job |
| Extra AWS cost | **$0** | A few cents per job |
| Needs S3 bucket | No | Yes |
| Needs IAM execution role with S3 access | No | Yes |
| Needs `ml.*` training quota | No | **Yes** |
| You learn | The ML concepts | The ML concepts **+ the SageMaker managed workflow** |

**If you are on a $1 budget, or a restricted sandbox account, use
Track B.** It produces the same model and the same numbers. Track A is
what the original AWS Academy lab does, and is worth showing if your
account and budget allow it.

In [ ]:
# ================= LAB CONFIGURATION =================
USE_MANAGED_SAGEMAKER = False    # <-- set True for the managed-job track

# Only used when USE_MANAGED_SAGEMAKER = True.
# These are the smallest instance types that SageMaker supports for each
# role, chosen to keep the cost down.
TRAIN_INSTANCE    = 'ml.m5.large'
ENDPOINT_INSTANCE = 'ml.t2.medium'
TRANSFORM_INSTANCE = 'ml.m5.large'
# =====================================================

import warnings; warnings.simplefilter('ignore')
import pandas as pd, numpy as np, os, json

print('Track:', 'A (managed SageMaker)' if USE_MANAGED_SAGEMAKER
      else 'B (in-notebook, no extra AWS cost)')

## Step 1 - Load the test set and the predictions

If `test_predictions.csv` from Lab 3.5 is missing, we regenerate it
locally so this notebook still runs.

In [ ]:
feature_cols = json.load(open('feature_cols.json'))
test = pd.read_csv('test.csv', header=None, names=['target'] + feature_cols)

if os.path.exists('test_predictions.csv'):
    probs = pd.read_csv('test_predictions.csv', header=None)[0].values
    print('Loaded predictions from Lab 3.5')
else:
    import xgboost as xgb
    booster = xgb.Booster(); booster.load_model('xgboost-model.json')
    probs = booster.predict(xgb.DMatrix(test[feature_cols]))
    print('Regenerated predictions from the saved model')

y_true = test['target'].values
y_prob = np.asarray(probs, dtype=float)
y_pred = (y_prob > 0.5).astype(int)

print('Test records:', len(y_true))
pd.DataFrame({'probability': y_prob.round(3),
              'predicted': y_pred,
              'actual': y_true}).head(10)

## Step 2 - The confusion matrix

The confusion matrix is the foundation every other metric is computed
from. With `Abnormal` as the positive class (1):

|  | Predicted Normal | Predicted Abnormal |
|---|---|---|
| **Actually Normal** | True Negative (TN) | False Positive (FP) |
| **Actually Abnormal** | False Negative (FN) | True Positive (TP) |

In this medical-screening framing:
* a **false positive** sends a healthy patient for an unnecessary
  follow-up - inconvenient and costly
* a **false negative** tells a patient with an abnormality that they
  are fine - clinically far more serious

That asymmetry is why accuracy alone is not enough.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print('Confusion matrix:')
print(pd.DataFrame(cm,
      index=['Actual Normal', 'Actual Abnormal'],
      columns=['Pred Normal', 'Pred Abnormal']))
print()
print(f'True  Negatives (TN): {tn:3d}  - healthy, correctly cleared')
print(f'False Positives (FP): {fp:3d}  - healthy, wrongly flagged')
print(f'False Negatives (FN): {fn:3d}  - abnormal, MISSED')
print(f'True  Positives (TP): {tp:3d}  - abnormal, correctly caught')

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['Normal','Abnormal']).plot(
    ax=ax, cmap='Blues', colorbar=False)
plt.title('Confusion matrix'); plt.tight_layout(); plt.show()

## Step 3 - Performance metrics

Each metric answers a different question:

| Metric | Formula | Question it answers |
|---|---|---|
| **Accuracy** | (TP+TN) / all | What fraction of all predictions were right? |
| **Precision** | TP / (TP+FP) | When it says Abnormal, how often is it right? |
| **Recall** (sensitivity) | TP / (TP+FN) | Of all truly Abnormal cases, how many did it catch? |
| **Specificity** | TN / (TN+FP) | Of all truly Normal cases, how many did it clear? |
| **F1** | harmonic mean of precision and recall | Single number balancing the two |

Compare accuracy against the **majority-class baseline** printed below.
A model that only just beats the baseline has learned very little.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report)

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred, zero_division=0)
spec = tn / (tn + fp) if (tn + fp) else 0.0
f1   = f1_score(y_true, y_pred, zero_division=0)
baseline = max(y_true.mean(), 1 - y_true.mean())

print(f'Accuracy    {acc:6.1%}')
print(f'Precision   {prec:6.1%}')
print(f'Recall      {rec:6.1%}   <- how many abnormal cases we caught')
print(f'Specificity {spec:6.1%}')
print(f'F1 score    {f1:6.1%}')
print()
print(f'Majority-class baseline accuracy: {baseline:.1%}')
print(f'Improvement over baseline:        {acc - baseline:+.1%}')

In [ ]:
print(classification_report(y_true, y_pred,
                            target_names=['Normal', 'Abnormal'],
                            zero_division=0))

## Step 4 - ROC curve and AUC

The 0.5 cutoff we used is a **choice**, not a property of the model.
The model outputs a probability; we decide where to draw the line.

The ROC curve sweeps that threshold from 0 to 1 and plots the
true-positive rate against the false-positive rate. **AUC** (area under
the curve) summarises it in one number:

* 0.5 = no better than random guessing
* 1.0 = perfect separation

AUC is threshold-independent, which makes it a fairer summary of the
model than accuracy on an imbalanced dataset.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, RocCurveDisplay

auc = roc_auc_score(y_true, y_prob)
fpr, tpr, thresholds = roc_curve(y_true, y_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, lw=2, label=f'XGBoost (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.500)')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve')
plt.legend(loc='lower right')
plt.tight_layout(); plt.show()

print(f'AUC: {auc:.4f}')

## Step 5 - Moving the threshold

Because false negatives are the costly error here, we may prefer a
threshold **below** 0.5: flag more patients as Abnormal, catching more
true cases at the price of more false alarms.

The table below shows that trade-off explicitly. There is no
mathematically "correct" row - the right choice depends on the relative
cost of the two errors, which is a business/clinical decision, not a
modelling one.

In [ ]:
rows = []
for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    p = (y_prob > t).astype(int)
    c = confusion_matrix(y_true, p, labels=[0, 1])
    tn_, fp_, fn_, tp_ = c.ravel()
    rows.append({
        'threshold': t,
        'accuracy':  round(accuracy_score(y_true, p), 3),
        'precision': round(precision_score(y_true, p, zero_division=0), 3),
        'recall':    round(recall_score(y_true, p, zero_division=0), 3),
        'f1':        round(f1_score(y_true, p, zero_division=0), 3),
        'missed_abnormal(FN)': fn_,
        'false_alarms(FP)': fp_,
    })

pd.DataFrame(rows).set_index('threshold')

## Step 6 - Record the baseline for Lab 3.7

Lab 3.7 tunes the hyperparameters and compares against these numbers,
so we save them.

In [ ]:
baseline_metrics = {'accuracy': float(acc), 'precision': float(prec),
                    'recall': float(rec), 'f1': float(f1), 'auc': float(auc)}
with open('baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2)
print(json.dumps(baseline_metrics, indent=2))

## A caution about this test set

The test set holds only about 31 records. One record changing sides
moves accuracy by roughly 3 percentage points. Treat these numbers as
indicative, not precise - and be sceptical of small differences between
models in Lab 3.7. With data this small, k-fold cross-validation is a
more trustworthy method than a single held-out split.

## Conclusion

You have:
* Used the test data to generate predictions
* Generated a confusion matrix
* Generated accuracy, precision, recall, specificity, F1 and AUC
* Seen how moving the decision threshold trades one error type for the other

**Remember to Stop the notebook instance when you are done.**

Next: `3_7-machinelearning.ipynb`